# LB-0016 — R3 vs. R2 Pro4-hint: SC16 + PAL3 + adaptive length

Creates two public-leaderboard CSVs **sequentially** with exactly the same frozen deployment pipeline: SC16 at 2,048 tokens, capped-candidate recovery at 4,096 then 8,192 tokens, and PAL4 with a ≥3 execution-agreement gate.

Use a fresh **A100** runtime. This is inference-only: it reads no answers and makes no external API calls. Cell 3 exposes the R3 adapter path explicitly; change only that one path if the desired R3 run differs.

In [ ]:
# Cell 1 — Fresh A100 only. Install, then restart the runtime once before Cell 2.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart runtime once now. Then run Cells 2–5 in order.")


In [ ]:
# Cell 2 — Mount Drive; clone/update the reproducibility repo; cache the pinned Qwen base.
# GITHUB_TOKEN is required only while the repo remains private. Store it in Colab Secrets, never in this notebook.
from google.colab import drive, userdata
from pathlib import Path
import subprocess
drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = "https://github.com/jhparktime/qwen-math-final-2026.git"
if token:
    url = "https://x-access-token:" + token + "@github.com/jhparktime/qwen-math-final-2026.git"
repo = Path("/content/qwen-math-final")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Freeze the two adapters and paired full-deployment configs.
import hashlib, json, re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r"[\s_-]+", "", unicodedata.normalize("NFC", str(value)).casefold())

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

roots = [p for p in Path("/content/drive/MyDrive").iterdir() if p.is_dir() and compact(p.name) == compact("2026소중한챌린지")]
assert len(roots) == 1, roots
project = roots[0]
runs = project / "runs"
leaderboard = project / "data" / "deep_chal_math_leaderboard_filtered.csv"
assert leaderboard.exists(), leaderboard

# Default R3 is the best historical R3 continuation. If you mean a different R3, change only this line.
R3_ADAPTER = runs / "RFT-0008D-r3mix-r2continue-r16-a100" / "adapter_final"
# Example for the newer public-data continuation instead: runs / "RFT-0015-r2pro4-external14k-lowdrift-r16" / "adapter_final"
R2_ADAPTER = runs / "RFT-0004B-r2-pro4-hint-lowdrift-lora" / "adapter_final"
adapters = {"r3": R3_ADAPTER, "r2_pro4hint": R2_ADAPTER}
for label, path in adapters.items():
    assert (path / "adapter_config.json").exists() and (path / "adapter_model.safetensors").exists(), (label, path)

base_config = json.loads((Path("configs") / "final_inference.json").read_text())
configs = {}
for label, adapter in adapters.items():
    config = json.loads(json.dumps(base_config))
    config["run_id"] = f"LB-0016-{label}-sc16-pal3-adaptive"
    config["expected_rows"] = 831
    config["model"]["adapter_name"] = label
    config["model"]["adapter_weight_sha256"] = sha256_file(adapter / "adapter_model.safetensors")
    config["adaptive_length"]["router"]["status"] = "full 2048→4096→8192 final deployment route enabled"
    config_path = Path("/content") / f"lb0016_{label}_config.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    configs[label] = {"adapter": adapter, "config": config_path, "output": runs / config["run_id"]}
print(json.dumps({k: {"adapter": str(v["adapter"]), "sha256": sha256_file(v["adapter"] / "adapter_model.safetensors"), "output": str(v["output"])} for k, v in configs.items()}, ensure_ascii=False, indent=2))


In [ ]:
# Cell 4 — Sequential paired inference. Resume-safe. Do not run another vLLM notebook in this runtime.
# Required order: R3 first, then R2 Pro4-hint.
for label in ["r3", "r2_pro4hint"]:
    item = configs[label]
    print(f"[RUN] {label}: SC16 + PAL3 + adaptive 4096/8192", flush=True)
    !PYTHONPATH=. python3 inference/final_inference.py --input {leaderboard} --adapter {item['adapter']} --output-dir {item['output']} --config {item['config']}
    !python3 scripts/validate_submission.py --input {leaderboard} --submission {item['output'] / 'submissions/submission.csv'} --expected-rows 831
    target = Path("/content/drive/MyDrive") / f"submission_{label}_sc16_pal3_adaptive.csv"
    target.write_bytes((item["output"] / "submissions/submission.csv").read_bytes())
    print("[SUBMIT]", target, flush=True)


In [ ]:
# Cell 5 — Summarize paired artifacts and hashes.
summary = {}
for label, item in configs.items():
    report = json.loads((item["output"] / "reports" / "inference_report.json").read_text())
    submission = item["output"] / "submissions" / "submission.csv"
    summary[label] = {
        "adapter": str(item["adapter"]),
        "submission": str(submission),
        "submission_sha256": sha256_file(submission),
        "adaptive_length_changes": report.get("summary", {}).get("adaptive_answer_changes"),
        "pal_changes": report.get("summary", {}).get("pal_answer_changes"),
    }
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Final cell — optional GPU release after both files and the summary are saved.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True when finished.")
